In [18]:
import pandas as pd
import numpy as np



In [ ]:
# Change this only if your file has a different name.
df = pd.read_csv("sweep_results_2026-04-29_16-56-49.csv")
df.head()


,param,value,model,record,f1,precision,recall,tp,fp,fn,threshold_updates
0,margin,0.05,Float32,100,0.970588,0.970588,0.970588,33,1,1,222
1,margin,0.05,Float32,103,0.000000,0.000000,0.000000,0,0,2,203
2,margin,0.05,Float32,105,0.430556,0.300971,0.756098,31,72,10,251
3,margin,0.05,Float32,111,0.166667,0.090909,1.000000,1,10,0,207
4,margin,0.05,Float32,113,0.238095,0.138889,0.833333,5,31,1,174


In [ ]:
summary = df.groupby(["model", "param", "value"], as_index=False).agg(
    records=("record", "nunique"),
    tp=("tp", "sum"),
    fp=("fp", "sum"),
    fn=("fn", "sum"),
    mean_f1=("f1", "mean"),
    mean_precision=("precision", "mean"),
    mean_recall=("recall", "mean"),
    mean_threshold_updates=("threshold_updates", "mean"),
)

summary["pooled_precision"] = np.where(
    summary["tp"] + summary["fp"] > 0,
    summary["tp"] / (summary["tp"] + summary["fp"]),
    0,
)

summary["pooled_recall"] = np.where(
    summary["tp"] + summary["fn"] > 0,
    summary["tp"] / (summary["tp"] + summary["fn"]),
    0,
)

summary["pooled_f1"] = np.where(
    summary["pooled_precision"] + summary["pooled_recall"] > 0,
    2
    * summary["pooled_precision"]
    * summary["pooled_recall"]
    / (summary["pooled_precision"] + summary["pooled_recall"]),
    0,
)

ranked = summary.sort_values(
    by=["model", "param", "pooled_f1", "pooled_recall", "pooled_precision"],
    ascending=[True, True, False, False, False],
)

top3_by_param = ranked.groupby(["model", "param"]).head(3).reset_index(drop=True)

top3_by_param[
    [
        "model",
        "param",
        "value",
        "records",
        "tp",
        "fp",
        "fn",
        "pooled_precision",
        "pooled_recall",
        "pooled_f1",
        "mean_threshold_updates",
    ]
]

,model,param,value,records,tp,fp,fn,pooled_precision,pooled_recall,pooled_f1,mean_threshold_updates
0,Float32,admit_ceil,0.95,22,3245,774,1811,0.807415,0.641812,0.715152,218.590909
1,Float32,admit_ceil,0.90,22,3260,895,1796,0.784597,0.644778,0.707849,218.590909
2,Float32,admit_ceil,0.85,22,3292,1026,1764,0.762390,0.651108,0.702368,218.590909
3,Float32,buffer_size,300.00,22,3291,1016,1765,0.764105,0.650910,0.702980,218.590909
4,Float32,buffer_size,200.00,22,3289,1015,1767,0.764173,0.650514,0.702778,218.590909
5,Float32,buffer_size,150.00,22,3292,1026,1764,0.762390,0.651108,0.702368,218.590909
6,Float32,ceil,0.99,22,3291,1005,1765,0.766061,0.650910,0.703807,218.590909
7,Float32,ceil,0.95,22,3292,1026,1764,0.762390,0.651108,0.702368,218.590909
8,Float32,ceil,0.90,22,3297,1159,1759,0.739901,0.652097,0.693230,218.590909
9,Float32,floor,0.60,22,3260,895,1796,0.784597,0.644778,0.707849,218.590909


In [26]:
top3_by_param.to_csv("top3_dynamic_by_param.csv", index=False)
print("Saved top3_dynamic_by_param.csv")

Saved top3_dynamic_by_param.csv
